# 04 — Embeddings et index sémantique FAISS

Construit la couche de **recherche sémantique** de Nayaar : chaque parfum
est encodé en vecteur (embedding) à partir de son `profil_text`, puis
indexé avec FAISS pour permettre une recherche par similarité de sens
(ex. "un parfum boisé et chaud pour l'hiver") plutôt que par mot-clé exact.

**Entrée** : `data/processed/nayaar_knowledge_base.csv` (colonne `profil_text`)

**Sorties** :
- `data/processed/nayaar_index.faiss` — l'index vectoriel
- `data/processed/nayaar_index_mapping.json` — correspondance position dans
  l'index → parfum (nom, marque, famille...), nécessaire car FAISS ne stocke
  que des vecteurs, pas de métadonnées

FAISS tourne **en local** (pas de base vectorielle externe au MVP, voir
`Docs/MVP_SCOPE.md`). Le modèle d'embeddings est `all-MiniLM-L6-v2`
(`sentence-transformers`) : léger (~80 Mo), rapide en local, et suffisant
pour un corpus de ~2000 phrases courtes.

Étapes :
1. Chargement de la Knowledge Base
2. Génération des embeddings
3. Construction et sauvegarde de l'index FAISS + mapping
4. Vérification qualité : 5 parfums, leurs 5 plus proches voisins, jugement olfactif

In [ ]:
import json

import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

CHEMIN_KNOWLEDGE_BASE = "../processed/nayaar_knowledge_base.csv"
CHEMIN_INDEX_FAISS = "../processed/nayaar_index.faiss"
CHEMIN_MAPPING = "../processed/nayaar_index_mapping.json"

NOM_MODELE = "all-MiniLM-L6-v2"  # 384 dimensions, léger, bon compromis qualité/vitesse


## 1. Chargement de la Knowledge Base

In [ ]:
df = pd.read_csv(CHEMIN_KNOWLEDGE_BASE, encoding="utf-8")

print(f"Parfums chargés : {len(df)}")
print(f"\nExemple de profil_text :\n{df['profil_text'].iloc[0]}")


## 2. Génération des embeddings

Chaque `profil_text` (nom, marque, famille, notes, profil, saison/moment
dominants — construit au notebook 03b) est encodé en un vecteur de 384
dimensions. On **normalise** les vecteurs (norme L2 = 1) : cela permet
ensuite d'utiliser une distance euclidienne dans FAISS qui se comporte
comme une distance cosinus, la métrique standard pour comparer des
embeddings de texte.

In [ ]:
modele = SentenceTransformer(NOM_MODELE)

textes = df["profil_text"].tolist()
embeddings = modele.encode(
    textes,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,  # norme L2 = 1, cf. explication ci-dessus
)

embeddings = embeddings.astype("float32")  # type attendu par FAISS
print(f"Forme des embeddings : {embeddings.shape}")  # (nb_parfums, 384)


## 3. Construction et sauvegarde de l'index FAISS

`IndexFlatL2` : recherche exacte (pas d'approximation), largement
suffisante pour ~2000 vecteurs — pas besoin d'un index approximatif (IVF,
HNSW...) à cette échelle. Le mapping position → parfum est sauvegardé à
part : FAISS ne connaît que des vecteurs numérotés 0, 1, 2..., pas les noms
de parfums.

In [ ]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print(f"Vecteurs indexés : {index.ntotal}")

faiss.write_index(index, CHEMIN_INDEX_FAISS)

mapping = [
    {"position": i, "nom": ligne["Name"], "marque": ligne["Brand"], "famille": ligne["famille"]}
    for i, ligne in df.iterrows()
]
with open(CHEMIN_MAPPING, "w", encoding="utf-8") as f:
    json.dump(mapping, f, ensure_ascii=False, indent=2)

print(f"Index sauvegardé : {CHEMIN_INDEX_FAISS}")
print(f"Mapping sauvegardé : {CHEMIN_MAPPING} ({len(mapping)} entrées)")


## 4. Vérification qualité

Pour 5 parfums tirés au hasard, on cherche leurs 5 plus proches voisins
dans l'index (on exclut le parfum lui-même, distance 0 avec lui-même). Le
but : juger **à l'œil** si la proximité sémantique correspond à une
proximité olfactive plausible (familles/notes similaires).

In [ ]:
def trouver_voisins(position_parfum, index, embeddings, mapping, k=5):
    """
    Cherche les k plus proches voisins d'un parfum déjà indexé (identifié par
    sa position). On demande k+1 voisins à FAISS car le parfum lui-même sera
    toujours son propre plus proche voisin (distance 0), qu'on retire ensuite.
    """
    vecteur_requete = embeddings[position_parfum : position_parfum + 1]
    distances, positions = index.search(vecteur_requete, k + 1)

    voisins = []
    for distance, position in zip(distances[0], positions[0]):
        if position == position_parfum:
            continue  # on exclut le parfum lui-même
        voisins.append({**mapping[position], "distance": round(float(distance), 4)})
    return voisins[:k]


parfums_a_tester = df.sample(5, random_state=42).index.tolist()

for position in parfums_a_tester:
    parfum = df.iloc[position]
    print(f"=== {parfum['Name']} ({parfum['Brand']}) — famille : {parfum['famille']} ===")
    print(f"Notes : {', '.join(json.loads(parfum['notes_list']))}\n")

    voisins = trouver_voisins(position, index, embeddings, mapping, k=5)
    for voisin in voisins:
        parfum_voisin = df[(df["Name"] == voisin["nom"]) & (df["Brand"] == voisin["marque"])].iloc[0]
        print(f"  - {voisin['nom']} ({voisin['marque']}) — famille : {voisin['famille']} — distance : {voisin['distance']}")
        print(f"    notes : {', '.join(json.loads(parfum_voisin['notes_list']))}")
    print()
